# Machine Learning Solution: NFL Big Data Bowl 2026
### Standalone Kaggle Notebook - 100% Self-Contained
- **Zero External Dependencies**: Does not require `tabular_pipeline` or custom pip installations.
- **Task Type**: `regression`
- **Target Column**: `x`
- **Evaluation Metric**: `RMSE`
- **Cross-Validation**: `5-Fold group_kfold`
- **Built-in Auto-FE & Stacking**: Includes universal interaction primitives and Out-Of-Fold Ridge stacking.


## 1. Environment Setup & Core Dependencies
Import standard data science packages available natively in Kaggle Docker environment.

In [ ]:
import os
import sys
import glob
import time
import json
import warnings
import re
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import KFold, StratifiedKFold, GroupKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import Ridge, LogisticRegression
from scipy.optimize import minimize

warnings.filterwarnings('ignore')

print(f'Pandas     : {pd.__version__}')
print(f'LightGBM   : {lgb.__version__}')
print(f'XGBoost    : {xgb.__version__}')
print(f'CatBoost   : {cb.__version__}')

## 2. Configuration & Path Resolution
Defines pipeline metadata and automatically locates datasets in `/kaggle/input` or local folders.

In [ ]:
CONFIG = {
    'target_col': 'x',
    'id_col': None,
    'group_col': 'play_id',
    'time_col': None,
    'drop_cols': ["y"],
    'task_type': 'regression',
    'eval_metric': 'rmse',
    'cv_strategy': 'group_kfold',
    'n_splits': 5,
    'seed': 2026,
    'models': ["lgbm", "xgb", "catboost"],
    'model_params': {},
    'train_path': 'cases/nfl_big_data_bowl_2026/train_stitched.csv',
    'test_path': 'cases/nfl_big_data_bowl_2026/test_stitched.csv'
}

# Protect secondary target coordinate from leaking into features
if CONFIG['target_col'] == 'x' and 'y' not in CONFIG['drop_cols']:
    CONFIG['drop_cols'].append('y')
elif CONFIG['target_col'] == 'y' and 'x' not in CONFIG['drop_cols']:
    CONFIG['drop_cols'].append('x')

def resolve_path(p: str) -> str:
    if not p:
        return ''
    if os.path.exists(p):
        return p
    if os.path.exists('/kaggle/input'):
        base = os.path.basename(p)
        matches = glob.glob(f'/kaggle/input/**/{base}', recursive=True)
        if matches:
            return matches[0]
        for f in glob.glob('/kaggle/input/**/*', recursive=True):
            if os.path.isfile(f) and os.path.basename(f).lower() == base.lower():
                return f
    base = os.path.basename(p)
    matches = glob.glob(f'./**/{base}', recursive=True)
    if matches:
        return matches[0]
    return p

def is_higher_better(metric_name: str) -> bool:
    return metric_name.lower() not in ['rmse', 'mae', 'logloss', 'loss', 'mse']

def calculate_metric(y_true, y_pred, metric_name, task_type='regression'):
    m = metric_name.lower()
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    if m == 'rmse':
        return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    elif m == 'mae':
        return float(np.mean(np.abs(y_true - y_pred)))
    elif m == 'roc_auc':
        from sklearn.metrics import roc_auc_score
        return float(roc_auc_score(y_true, y_pred))
    elif m == 'logloss':
        from sklearn.metrics import log_loss
        p = np.clip(y_pred, 1e-15, 1 - 1e-15)
        return float(log_loss(y_true, p))
    elif m == 'accuracy':
        from sklearn.metrics import accuracy_score
        pred_labels = (y_pred >= 0.5).astype(int) if task_type == 'binary' else np.argmax(y_pred, axis=1)
        return float(accuracy_score(y_true, pred_labels))
    elif m == 'f1_macro':
        from sklearn.metrics import f1_score
        pred_labels = (y_pred >= 0.5).astype(int) if task_type == 'binary' else np.argmax(y_pred, axis=1)
        return float(f1_score(y_true, pred_labels, average='macro'))
    elif m == 'r2':
        from sklearn.metrics import r2_score
        return float(r2_score(y_true, y_pred))
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


## 3. Data Ingestion & Partition Stitching Fallback
Loads the preprocessed dataset, or stitches raw competition files if the preprocessed file is not attached.

In [ ]:
train_path = resolve_path(CONFIG['train_path'])
train_df = None
if os.path.exists(train_path):
    print(f'Loading train dataset from: {train_path}')
    train_df = pd.read_csv(train_path)
else:
    stitched_cands = glob.glob('/kaggle/input/**/train_stitched.csv', recursive=True) or glob.glob('/kaggle/input/**/train_merged.csv', recursive=True)
    if stitched_cands:
        print(f'Found pre-stitched dataset: {stitched_cands[0]}')
        train_df = pd.read_csv(stitched_cands[0])
    else:
        output_files = sorted(glob.glob('/kaggle/input/**/output_*.csv', recursive=True)) or sorted(glob.glob('./**/output_*.csv', recursive=True))
        input_files = sorted(glob.glob('/kaggle/input/**/input_*.csv', recursive=True)) or sorted(glob.glob('./**/input_*.csv', recursive=True))
        if output_files and input_files:
            print(f'[Auto-Stitch Partitions] Detected {len(output_files)} target partition files and {len(input_files)} feature store partition files.')
            print('Concatenating output target partitions (outputs)...')
            base_df = pd.concat([pd.read_csv(f) for f in output_files], ignore_index=True)
            print('Concatenating input feature partitions (inputs)...')
            join_df = pd.concat([pd.read_csv(f) for f in input_files], ignore_index=True)
            print("Merging target labels with feature store on ['game_id', 'play_id', 'nfl_id']...")
            p_feat = join_df.drop_duplicates(subset=['game_id', 'play_id', 'nfl_id']).copy()
            p_feat = p_feat.rename(columns={'x': 'x_feature', 'y': 'y_feature'})
            if 'frame_id' in p_feat.columns:
                p_feat = p_feat.drop(columns=['frame_id'])
            train_df = pd.merge(base_df, p_feat, on=['game_id', 'play_id', 'nfl_id'], how='inner')
            print(f'[Auto-Stitch Complete] Stitched training matrix: {train_df.shape[0]:,} rows x {train_df.shape[1]} columns')
        else:
            generic_train = resolve_path('train.csv')
            if generic_train and os.path.exists(generic_train):
                print(f'Loading generic train.csv from: {generic_train}')
                train_df = pd.read_csv(generic_train)
            else:
                csv_list = glob.glob('/kaggle/input/**/*.csv', recursive=True)
                print(f'Available CSV files in /kaggle/input: {csv_list}')
                raise FileNotFoundError('Training dataset could not be located in /kaggle/input or local directories. Please attach the dataset to your Kaggle notebook.')

test_path = resolve_path(CONFIG['test_path'])
test_df = None
if test_path and os.path.exists(test_path):
    print(f'Loading test dataset from: {test_path}')
    test_df = pd.read_csv(test_path)
else:
    raw_test_file = resolve_path('test.csv')
    raw_test_in_file = resolve_path('test_input.csv')
    if raw_test_file and os.path.exists(raw_test_file) and raw_test_in_file and os.path.exists(raw_test_in_file):
        print(f'Auto-stitching test features: {raw_test_file} + {raw_test_in_file}...')
        raw_test = pd.read_csv(raw_test_file)
        raw_test_in = pd.read_csv(raw_test_in_file)
        p_feat = raw_test_in.drop_duplicates(subset=['game_id', 'play_id', 'nfl_id']).copy()
        p_feat = p_feat.rename(columns={'x': 'x_feature', 'y': 'y_feature'})
        if 'frame_id' in p_feat.columns:
            p_feat = p_feat.drop(columns=['frame_id'])
        test_df = pd.merge(raw_test, p_feat, on=['game_id', 'play_id', 'nfl_id'], how='left')

print(f'Training Matrix : {train_df.shape[0]:,} rows x {train_df.shape[1]} columns')
if test_df is not None:
    print(f'Testing Matrix  : {test_df.shape[0]:,} rows x {test_df.shape[1]} columns')
train_df.head(3)

## 4. Feature Engineering & Universal Auto-FE Engine
Executes custom hooks plus automated domain interactions (pairwise categoricals, frequencies, and group mean deviations).

In [ ]:
class BaseFeatureHook:
    def transform(self, train_df: pd.DataFrame, test_df: pd.DataFrame = None):
        return train_df, test_df

class CustomFeatures(BaseFeatureHook):
    def transform(self, train_df: pd.DataFrame, test_df: pd.DataFrame = None):
        return train_df, test_df

def apply_auto_feature_engineering(train_df: pd.DataFrame, test_df: pd.DataFrame = None):
    print('[Auto-FE Engine] Generating domain interaction primitives...')
    train_out = train_df.copy()
    test_out = test_df.copy() if test_df is not None else None
    dfs = [train_out] if test_out is None else [train_out, test_out]

    target_col = CONFIG.get('target_col')
    drop_cols = set(CONFIG.get('drop_cols', []) + [target_col])
    if CONFIG.get('id_col'):
        drop_cols.add(CONFIG.get('id_col'))
    if CONFIG.get('group_col'):
        drop_cols.add(CONFIG.get('group_col'))

    candidate_cols = [c for c in train_out.columns if c not in drop_cols]
    num_cols = [c for c in candidate_cols if pd.api.types.is_numeric_dtype(train_out[c]) and train_out[c].nunique() > 2]
    cat_cols = [c for c in candidate_cols if c not in num_cols]

    new_feats = []
    # 1. High-Cardinality Frequency Encoding (nunique > 15)
    for col in cat_cols:
        if train_out[col].nunique() > 15:
            freq_map = train_out[col].value_counts(normalize=True).to_dict()
            col_name = f'{col}_freq'
            new_feats.append(col_name)
            for df in dfs:
                if col in df.columns:
                    df[col_name] = df[col].map(freq_map).fillna(0.0)

    # 2. Pairwise Categorical Interactions (Cardinality 2-25)
    informative_cats = [c for c in cat_cols if 2 <= train_out[c].nunique() <= 25]
    if len(informative_cats) >= 2:
        top_pairs = []
        for i in range(min(3, len(informative_cats))):
            for j in range(i + 1, min(4, len(informative_cats))):
                top_pairs.append((informative_cats[i], informative_cats[j]))
        for c1, c2 in top_pairs[:4]:
            pair_name = f'{c1}_x_{c2}'
            freq_pair_name = f'{pair_name}_freq'
            new_feats.extend([pair_name, freq_pair_name])
            for df in dfs:
                if c1 in df.columns and c2 in df.columns:
                    df[pair_name] = df[c1].astype(str) + '_' + df[c2].astype(str)
            freq_map = train_out[pair_name].value_counts(normalize=True).to_dict()
            for df in dfs:
                df[freq_pair_name] = df[pair_name].map(freq_map).fillna(0.0)

    # 3. Group-by Numerical Mean Differences
    if len(cat_cols) > 0 and len(num_cols) > 0:
        candidate_cats = [c for c in cat_cols if 2 <= train_out[c].nunique() <= 50]
        agg_cats = candidate_cats[:2]
        agg_nums = num_cols[:3]
        for cat in agg_cats:
            for num in agg_nums:
                grp = train_out.groupby(cat)[num].mean().to_dict()
                diff_col = f'{num}_diff_{cat}'
                new_feats.append(diff_col)
                for df in dfs:
                    if cat in df.columns and num in df.columns:
                        df[diff_col] = df[num] - df[cat].map(grp).fillna(0.0)

    # 4. Prune collinear / near-constant features
    all_created = [c for c in new_feats if c in train_out.columns]
    if all_created:
        num_created = [c for c in all_created if np.issubdtype(train_out[c].dtype, np.number)]
        valid_num = []
        if num_created:
            variances = train_out[num_created].var()
            valid_num = variances[variances > 1e-7].index.tolist()
            if len(valid_num) > 1:
                corr_matrix = train_out[valid_num].corr().abs()
                to_drop = set()
                for i in range(len(valid_num)):
                    for j in range(i + 1, len(valid_num)):
                        f1, f2 = valid_num[i], valid_num[j]
                        if f1 not in to_drop and f2 not in to_drop and corr_matrix.loc[f1, f2] >= 0.98:
                            to_drop.add(f2)
                valid_num = [f for f in valid_num if f not in to_drop]
        non_num_created = [c for c in all_created if not np.issubdtype(train_out[c].dtype, np.number)]
        kept_feats = (valid_num + non_num_created)[:40]
        drop_candidates = [c for c in all_created if c not in kept_feats]
        for df in dfs:
            df.drop(columns=[c for c in drop_candidates if c in df.columns], inplace=True)

    # 5. Sanitize column names for GBDT
    clean_map = {c: re.sub(r'[^a-zA-Z0-9_]+', '_', str(c)) for c in train_out.columns}
    train_out = train_out.rename(columns=clean_map)
    if test_out is not None:
        test_out = test_out.rename(columns=clean_map)

    active_f = [c for c in train_out.columns if c not in drop_cols]
    print(f'[Auto-FE Complete] Total features active: {len(active_f)} cols.')
    return train_out, test_out

# Execute Feature Engineering
hook = CustomFeatures()
train_df, test_df = hook.transform(train_df, test_df)
train_df, test_df = apply_auto_feature_engineering(train_df, test_df)
print('Feature engineering pipeline executed successfully.')

## 5. Leak-Free Cross-Validation Training
Executes 5-Fold CV across LightGBM, XGBoost, and CatBoost with early stopping and leak-free categorical encoding.

In [ ]:
target_col = CONFIG['target_col']
id_col = CONFIG['id_col']
group_col = CONFIG['group_col']
task_type = CONFIG['task_type']
eval_metric = CONFIG['eval_metric']
n_splits = CONFIG['n_splits']
seed = CONFIG['seed']

y_train = train_df[target_col].values.astype(float if task_type == 'regression' else int)

all_drop = set(CONFIG['drop_cols'] + [target_col])
if id_col:
    all_drop.add(id_col)
if group_col:
    all_drop.add(group_col)

feature_cols = [c for c in train_df.columns if c not in all_drop]
X_train = train_df[feature_cols].copy()

X_test = None
if test_df is not None:
    matching = [c for c in feature_cols if c in test_df.columns]
    X_test = test_df[feature_cols].copy() if len(matching) == len(feature_cols) else test_df.reindex(columns=feature_cols).copy()

cat_cols = [c for c in feature_cols if X_train[c].dtype == 'object' or isinstance(X_train[c].dtype, pd.CategoricalDtype)]
print(f'Training Features : {len(feature_cols)} features ({len(cat_cols)} categorical)')

if group_col and group_col in train_df.columns:
    print(f'Using GroupKFold on column {group_col} ({n_splits} splits)')
    splitter = GroupKFold(n_splits=n_splits)
    splits = list(splitter.split(X_train, y_train, groups=train_df[group_col].values))
elif task_type in ['binary', 'multiclass']:
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = list(splitter.split(X_train, y_train))
else:
    splitter = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = list(splitter.split(X_train, y_train))

model_names = CONFIG['models']
oof_predictions = {m: np.zeros(len(X_train)) for m in model_names}
test_predictions = {m: np.zeros(len(X_test)) if X_test is not None else None for m in model_names}
single_scores = {}

for m_name in model_names:
    print(f'\nTraining {m_name.upper()} ({n_splits} Folds)...')
    user_params = CONFIG['model_params'].get(m_name, {})

    for fold, (train_idx, val_idx) in enumerate(splits):
        X_tr, y_tr = X_train.iloc[train_idx].copy(), y_train[train_idx]
        X_va, y_val = X_train.iloc[val_idx].copy(), y_train[val_idx]
        X_te = X_test.copy() if X_test is not None else None

        if cat_cols:
            enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            X_tr[cat_cols] = enc.fit_transform(X_tr[cat_cols].astype(str))
            X_va[cat_cols] = enc.transform(X_va[cat_cols].astype(str))
            if X_te is not None:
                X_te[cat_cols] = enc.transform(X_te[cat_cols].astype(str))

        if m_name == 'lgbm':
            params = {'n_estimators': 3000, 'learning_rate': 0.03, 'num_leaves': 45, 'subsample': 0.8, 'colsample_bytree': 0.75, 'random_state': seed + fold, 'n_jobs': -1, 'verbose': -1}
            params.update(user_params)
            model = lgb.LGBMRegressor(**params) if task_type == 'regression' else lgb.LGBMClassifier(**params)
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_val)], callbacks=[lgb.early_stopping(100, verbose=False)])
        elif m_name == 'xgb':
            params = {'n_estimators': 3000, 'learning_rate': 0.03, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 0.75, 'random_state': seed + fold, 'n_jobs': -1, 'early_stopping_rounds': 100}
            params.update(user_params)
            model = xgb.XGBRegressor(**params) if task_type == 'regression' else xgb.XGBClassifier(**params, eval_metric='logloss')
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_val)], verbose=False)
        elif m_name == 'catboost':
            params = {'iterations': 3000, 'learning_rate': 0.03, 'depth': 6, 'random_seed': seed + fold, 'thread_count': -1, 'early_stopping_rounds': 100, 'verbose': False}
            params.update(user_params)
            model = cb.CatBoostRegressor(**params) if task_type == 'regression' else cb.CatBoostClassifier(**params)
            model.fit(X_tr, y_tr, eval_set=(X_va, y_val), verbose=False)

        if task_type == 'binary':
            val_preds = model.predict_proba(X_va)[:, 1]
            te_preds = model.predict_proba(X_te)[:, 1] if X_te is not None else None
        else:
            val_preds = model.predict(X_va)
            te_preds = model.predict(X_te) if X_te is not None else None

        oof_predictions[m_name][val_idx] = val_preds
        if X_te is not None:
            test_predictions[m_name] += te_preds / n_splits

        f_score = calculate_metric(y_val, val_preds, eval_metric, task_type)
        print(f'  Fold {fold + 1}/{n_splits} {eval_metric.upper()}: {f_score:.5f}')

    sc = calculate_metric(y_train, oof_predictions[m_name], eval_metric, task_type)
    single_scores[m_name] = sc
    print(f'=> {m_name.upper()} Overall OOF {eval_metric.upper()}: {sc:.5f}')

## 6. Multi-Model Ensembling & Validation Leaderboard
Blends predictions using Equal Mean, Scipy SLSQP Weight Optimization, and Out-Of-Fold Ridge Stacking.

In [ ]:
ensemble_scores = dict(single_scores)
all_oof = dict(oof_predictions)
all_test = dict(test_predictions)

if len(model_names) > 1:
    # Strategy A: Simple Average
    mean_oof = np.mean([oof_predictions[m] for m in model_names], axis=0)
    ensemble_scores['ensemble_mean'] = calculate_metric(y_train, mean_oof, eval_metric, task_type)
    all_oof['ensemble_mean'] = mean_oof
    if X_test is not None:
        all_test['ensemble_mean'] = np.mean([test_predictions[m] for m in model_names], axis=0)

    # Strategy B: Scipy SLSQP Weight Optimization
    def loss_fn(weights):
        weights = np.array(weights) / np.sum(weights)
        blended = sum(w * oof_predictions[m] for w, m in zip(weights, model_names))
        score = calculate_metric(y_train, blended, eval_metric, task_type)
        return -score if is_higher_better(eval_metric) else score

    opt_res = minimize(loss_fn, [1.0 / len(model_names)] * len(model_names), bounds=[(0, 1)] * len(model_names), method='SLSQP')
    opt_w = opt_res.x / np.sum(opt_res.x)
    opt_oof = sum(w * oof_predictions[m] for w, m in zip(opt_w, model_names))
    ensemble_scores['ensemble_opt_weights'] = calculate_metric(y_train, opt_oof, eval_metric, task_type)
    all_oof['ensemble_opt_weights'] = opt_oof
    if X_test is not None:
        all_test['ensemble_opt_weights'] = sum(w * test_predictions[m] for w, m in zip(opt_w, model_names))

    # Strategy C: Out-Of-Fold Stacking Meta-Learner
    X_meta_train = np.column_stack([oof_predictions[m] for m in model_names])
    X_meta_test = np.column_stack([test_predictions[m] for m in model_names]) if X_test is not None else None
    if task_type in ['binary', 'multiclass']:
        meta_learner = LogisticRegression(random_state=seed, max_iter=500)
        meta_learner.fit(X_meta_train, y_train)
        stack_oof = meta_learner.predict_proba(X_meta_train)[:, 1] if task_type == 'binary' else meta_learner.predict_proba(X_meta_train)
        stack_test = meta_learner.predict_proba(X_meta_test)[:, 1] if X_meta_test is not None else None
    else:
        meta_learner = Ridge(alpha=1.0, random_state=seed)
        meta_learner.fit(X_meta_train, y_train)
        stack_oof = meta_learner.predict(X_meta_train)
        stack_test = meta_learner.predict(X_meta_test) if X_meta_test is not None else None

    ensemble_scores['ensemble_stacking'] = calculate_metric(y_train, stack_oof, eval_metric, task_type)
    all_oof['ensemble_stacking'] = stack_oof
    if X_test is not None:
        all_test['ensemble_stacking'] = stack_test

higher_better = is_higher_better(eval_metric)
sorted_strategies = sorted(ensemble_scores.items(), key=lambda x: x[1], reverse=higher_better)
champion_name, champion_score = sorted_strategies[0]

leaderboard = pd.DataFrame(sorted_strategies, columns=['Strategy', f'OOF_{eval_metric.upper()}'])
leaderboard.insert(0, 'Rank', np.arange(1, len(leaderboard) + 1))
print(f'\nCHAMPION MODEL: {champion_name} (Score: {champion_score:.5f})')
leaderboard

## 7. Generate Submission File
Writes `submission.csv` to current working directory (`/kaggle/working/submission.csv`), fully compliant with Kaggle submission protocols.

In [ ]:
if test_df is not None and all_test.get(champion_name) is not None:
    champ_preds = all_test[champion_name]
    sub = pd.DataFrame()
    if id_col and id_col in test_df.columns:
        sub[id_col] = test_df[id_col]
    elif 'id' in test_df.columns:
        sub['id'] = test_df['id']
    elif 'ID' in test_df.columns:
        sub['ID'] = test_df['ID']
    else:
        sub['id'] = np.arange(len(test_df))

    sub[target_col] = champ_preds
    sub.to_csv('submission.csv', index=False)
    print(f'Submission successfully generated! Shape: {sub.shape}')
    print(f'Missing values: {sub.isna().sum().sum()}')
    print(sub.head(10))
else:
    print('No test set provided or test inference was skipped.')